# agentv25_langsmith_evaluation\n\nThis version introduces local regression evaluation and optional LangSmith dataset upload.\n

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

cwd = Path.cwd()
load_dotenv(cwd / ".env")
load_dotenv(cwd.parent / ".env")

print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith project:", os.getenv("LANGSMITH_PROJECT"))
print("LangSmith key present:", bool(os.getenv("LANGSMITH_API_KEY")))\n

## Define graph under test\n

In [ ]:
from typing import NotRequired, TypedDict
from langgraph.graph import START, END, StateGraph

class AgentState(TypedDict):
    input: str
    classification: NotRequired[str]
    answer: NotRequired[str]

def classify_node(state: AgentState) -> AgentState:
    text = state["input"].lower()
    if any(term in text for term in ["timeout", "latency", "incident", "error", "spike"]):
        classification = "incident_analysis"
    elif any(term in text for term in ["metric", "metrics", "count", "volume", "by command", "by client"]):
        classification = "analytics"
    else:
        classification = "general"
    return {"classification": classification}

def answer_node(state: AgentState) -> AgentState:
    c = state["classification"]
    if c == "incident_analysis":
        answer = "This is an EPP incident analysis request. For CHECK-DOMAIN after R13, inspect CONNECTION_TIMEOUT, elevated latency, failure concentration by client, registry connectivity, DNS resolver latency, and connection pool saturation."
    elif c == "analytics":
        answer = "This is an EPP analytics request. Analyze metrics by command, client, failure reason, volume, response_time, and grouped failures by client and command."
    else:
        answer = "This agent is used for EPP SLA analysis, including incidents, metrics, latency, failure volume, release impact, and operational recommendations."
    return {"answer": answer}

def build_graph():
    g = StateGraph(AgentState)
    g.add_node("classify", classify_node)
    g.add_node("answer", answer_node)
    g.add_edge(START, "classify")
    g.add_edge("classify", "answer")
    g.add_edge("answer", END)
    return g.compile()

graph = build_graph()
graph\n

## Define evaluation cases\n

In [ ]:
eval_cases = [
    {"case_id":"case_001","input":"Investigate CHECK-DOMAIN timeout spike after release R13.","expected_classification":"incident_analysis","required_terms":["CHECK-DOMAIN","CONNECTION_TIMEOUT","R13"],"forbidden_terms":["AUTH_FAILED","payment"]},
    {"case_id":"case_002","input":"Summarize EPP volume metrics by command.","expected_classification":"analytics","required_terms":["metrics","command"],"forbidden_terms":["rollback","payment"]},
    {"case_id":"case_003","input":"What is this agent used for?","expected_classification":"general","required_terms":["EPP"],"forbidden_terms":["database corruption","security breach"]},
]\n

## Evaluators\n

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class EvalResult:
    name: str
    passed: bool
    score: float
    notes: str

def normalize(text: str) -> str:
    return text.lower().strip()

def classification_evaluator(output: dict[str, Any], case: dict[str, Any]) -> EvalResult:
    actual = output.get("classification")
    expected = case.get("expected_classification")
    passed = actual == expected
    return EvalResult("classification", passed, 1.0 if passed else 0.0, f"expected={expected}, actual={actual}")

def required_terms_evaluator(output: dict[str, Any], case: dict[str, Any]) -> EvalResult:
    answer = normalize(output.get("answer", ""))
    terms = case.get("required_terms", [])
    missing = [term for term in terms if normalize(term) not in answer]
    score = 1.0 if not terms else (len(terms) - len(missing)) / len(terms)
    return EvalResult("required_terms", not missing, score, f"missing={missing}")

def forbidden_terms_evaluator(output: dict[str, Any], case: dict[str, Any]) -> EvalResult:
    answer = normalize(output.get("answer", ""))
    terms = case.get("forbidden_terms", [])
    found = [term for term in terms if normalize(term) in answer]
    return EvalResult("forbidden_terms", not found, 1.0 if not found else 0.0, f"found={found}")

def evaluate_case(output: dict[str, Any], case: dict[str, Any]) -> dict[str, Any]:
    evals = [classification_evaluator(output, case), required_terms_evaluator(output, case), forbidden_terms_evaluator(output, case)]
    score = 0.4 * evals[0].score + 0.4 * evals[1].score + 0.2 * evals[2].score
    return {"case_id": case["case_id"], "passed": all(e.passed for e in evals), "score": round(score, 3), "evaluators": evals}\n

## Run evaluation\n

In [ ]:
results = []
for case in eval_cases:
    output = graph.invoke({"input": case["input"]})
    result = evaluate_case(output, case)
    results.append({"case": case, "output": output, "eval": result})

for item in results:
    print(item["case"]["case_id"], "passed=", item["eval"]["passed"], "score=", item["eval"]["score"])
    print("classification:", item["output"]["classification"])
    print("answer:", item["output"]["answer"])
    for ev in item["eval"]["evaluators"]:
        print(" -", ev.name, ev.passed, ev.score, ev.notes)
    print()

overall = sum(item["eval"]["score"] for item in results) / len(results)
print("Overall score:", round(overall, 3))\n

## Visualize graph\n

In [ ]:
print(graph.get_graph().draw_mermaid())\n